# Notebook 9 — Building Knowledge Graphs from Biological Tables (Final)
Goal: Construct a small biological knowledge graph from tabular datasets using `pandas` and `networkx`, while demonstrating realistic practices such as identifier normalization and typed nodes.

## Section 1 — Import Libraries

In [ ]:
import pandas as pd
import networkx as nx

## Section 2 — Example Biological Tables with Stable Identifiers

In [ ]:
gene_pathway = pd.DataFrame({
    "gene_id": [
        "HGNC:11998",
        "HGNC:6407",
        "HGNC:1097",
        "HGNC:3236",
        "HGNC:8975"
    ],
    "gene_symbol": [
        "TP53",
        "KRAS",
        "BRAF",
        "EGFR",
        "PIK3CA"
    ],
    "pathway": [
        "Cell cycle",
        "MAPK pathway",
        "MAPK pathway",
        "EGFR signaling",
        "PI3K pathway"
    ]
})

drug_target = pd.DataFrame({
    "drug": ["Trametinib", "Cetuximab", "Erlotinib", "Alpelisib"],
    "target_gene_id": ["HGNC:6840", "HGNC:3236", "HGNC:3236", "HGNC:8975"]
})

patient_mutation = pd.DataFrame({
    "patient": ["P1","P2","P3","P4","P5"],
    "gene_id": [
        "HGNC:6407",
        "HGNC:11998",
        "HGNC:1097",
        "HGNC:3236",
        "HGNC:8975"
    ]
})

gene_pathway, drug_target, patient_mutation

## Section 3 — Create Graph

In [ ]:
G = nx.Graph()
G

## Section 4 — Normalize Entities and Create Typed Nodes

In [ ]:
# Gene nodes with stable identifiers
for _, row in gene_pathway.iterrows():
    G.add_node(
        row["gene_id"],
        type="gene",
        symbol=row["gene_symbol"]
    )

# Pathway nodes
for pathway in gene_pathway["pathway"].unique():
    G.add_node(pathway, type="pathway")

# Drug nodes
for drug in drug_target["drug"].unique():
    G.add_node(drug, type="drug")

# Patient nodes
for patient in patient_mutation["patient"].unique():
    G.add_node(patient, type="patient")

list(G.nodes(data=True))[:10]

## Section 5 — Add Gene–Pathway Relationships

In [ ]:
for _, row in gene_pathway.iterrows():
    G.add_edge(
        row["gene_id"],
        row["pathway"],
        relationship="participates_in"
    )

list(G.edges(data=True))[:10]

## Section 6 — Add Drug–Target Relationships

In [ ]:
for _, row in drug_target.iterrows():
    G.add_edge(
        row["drug"],
        row["target_gene_id"],
        relationship="targets"
    )

list(G.edges(data=True))

## Section 7 — Add Patient Mutation Relationships

In [ ]:
for _, row in patient_mutation.iterrows():
    G.add_edge(
        row["patient"],
        row["gene_id"],
        relationship="has_mutation"
    )

list(G.edges(data=True))

## Section 8 — Query by Node Type

In [ ]:
genes = [n for n,d in G.nodes(data=True) if d.get("type") == "gene"]
genes

## Section 9 — Query by Relationship Type

In [ ]:
mutation_edges = [
    (u,v,d) for u,v,d in G.edges(data=True)
    if d.get("relationship") == "has_mutation"
]

mutation_edges

## Section 10 — Simple Biological Questions

In [ ]:
# Which patients have KRAS mutations?

kras_id = "HGNC:6407"

[p for p in G.neighbors(kras_id)
 if G.nodes[p].get("type") == "patient"]

In [ ]:
# Which pathways is KRAS connected to?

[n for n in G.neighbors(kras_id)
 if G.nodes[n].get("type") == "pathway"]

## Section 11 — Export Graph as Triples

In [ ]:
triples = [
    (u, d["relationship"], v)
    for u,v,d in G.edges(data=True)
]

triples[:10]

## Section 12 — Build from CSV Files

In [ ]:
gene_pathway.to_csv("gene_pathway.csv", index=False)
drug_target.to_csv("drug_target.csv", index=False)
patient_mutation.to_csv("patient_mutation.csv", index=False)

gp = pd.read_csv("gene_pathway.csv")
dt = pd.read_csv("drug_target.csv")
pm = pd.read_csv("patient_mutation.csv")

gp.head(), dt.head(), pm.head()

## Skills Gained
- constructing graphs from biological tables
- entity normalization using stable identifiers
- creating typed nodes and relationship edges
- querying graph structure
- exporting graphs to triple format
- understanding the ingestion workflow used in many knowledge graph pipelines